In [ ]:
# import packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import liana as li

from liana.method import cellphonedb, connectome

from pathlib import Path
#import session_info

In [ ]:
adata_8um_original = sc.read_h5ad('../Visium/HumanKidney_Visium_HD_8um.h5ad')

In [ ]:
adata_8um_original

In [ ]:
adata = sc.read("./VisiumHD_NOTwithref_annoated.h5ad")

In [ ]:
adata_sub = adata[(adata.obs['x'] >= 16000) * (adata.obs['x'] <= 24000) * (adata.obs['y'] >= 26000) * (adata.obs['y'] <= 29000), :].copy()
adata_sub

In [ ]:
def match_spot(adata, adata_origin, cutoff):
    selected_spots=[]
    dist_x = []
    dist_y =[]
    for i in range(adata.obsm['spatial'].shape[0]):
        tmp_dist = (adata_origin.obsm['spatial'][:, 0] - adata.obs['x'][i])**2 + (adata_origin.obsm['spatial'][:, 1] - adata.obs['y'][i])**2
        tmp_index = np.where(tmp_dist == np.min(tmp_dist))[0][0]
        selected_spots.append(tmp_index)
        dist_x.append(np.abs(adata_origin.obsm['spatial'][tmp_index, 0] - adata.obs['x'][i]))
        dist_y.append(np.abs(adata_origin.obsm['spatial'][tmp_index, 1] - adata.obs['y'][i]))
    cut_off=cutoff    
    matched_spots = (np.array(dist_x)<cut_off)*(np.array(dist_y)<cut_off)
    final_selected_spots = np.array(selected_spots)[matched_spots]
    
    return matched_spots, final_selected_spots

In [ ]:
adata_sub.uns['spatial']

In [ ]:
matched_spots, final_selected_spots = match_spot(adata_sub, adata_8um_original, 15)

In [ ]:
np.sum(matched_spots)

In [ ]:
adata_8um_original_sub = adata_8um_original[final_selected_spots, adata_sub.var_names].copy()

In [ ]:
plt.scatter(adata.obs['x'], adata.obs['y'], s = 0.01)
plt.scatter(adata_8um_original_sub.obsm['spatial'][:, 0], adata_8um_original_sub.obsm['spatial'][:, 1], s = 0.002)

In [ ]:
adata_8um_original_sub.layers['counts'] = adata_8um_original_sub.X.copy()

In [ ]:
adata_8um_original_sub.X =adata_8um_original_sub.layers['counts'].copy()
sc.pp.calculate_qc_metrics(adata_8um_original_sub, log1p=False, inplace=True)

In [ ]:
plt.hist(adata_8um_original_sub.obs['total_counts'])

In [ ]:
plt.hist(adata_8um_original_sub.obs['total_counts'])

In [ ]:
np.sum(adata_8um_original_sub.obs['total_counts']==0)

In [ ]:
adata_sub

In [ ]:
adata_8um_original_sub.obs['celltype'] = np.array(adata_sub.obs['celltype_final']).copy()

In [ ]:
adata_8um_original_sub2 = adata_8um_original_sub[adata_8um_original_sub.obs['total_counts']>0, :].copy()
adata_8um_original_sub2

In [ ]:
adata_8um_original_sub2.X =adata_8um_original_sub2.layers['counts'].copy()
sc.pp.normalize_total(adata_8um_original_sub2, target_sum=1e4)
sc.pp.log1p(adata_8um_original_sub2)
#sc.pp.scale(adata_8um_original_sub2, max_value=10)

In [ ]:
import decoupler as dc
# obtain genesets
progeny = dc.op.progeny(organism='human', top=500)

In [ ]:
from liana.method import MistyData, genericMistyData, lrMistyData

In [ ]:
# use multivariate linear model to estimate activity
dc.mt.mlm(
    adata_8um_original_sub2,
    net=progeny,
    verbose=True,
    raw=False
)

In [ ]:
acts_progeny = li.ut.obsm_to_adata(adata_8um_original_sub2, 'score_mlm')

In [ ]:
adata_8um_original_sub2.obsm['score_mlm']

In [ ]:
acts_progeny

In [ ]:
acts_progeny.obsm['spatial'] = adata_sub.obsm['spatial'][adata_8um_original_sub.obs['total_counts']>0, :]

In [ ]:
acts_progeny.obsm['spatial']

In [ ]:
acts_progeny.obs['EGFR_mlm'] = acts_progeny.obsm['score_mlm'].iloc[:, 1]
acts_progeny.obs['PI3K_mlm'] = acts_progeny.obsm['score_mlm'].iloc[:, 7]
acts_progeny.obs['Hypoxia_mlm'] = acts_progeny.obsm['score_mlm'].iloc[:, 3]
acts_progeny.obs['JAK-STAT_mlm'] = acts_progeny.obsm['score_mlm'].iloc[:, 4]
acts_progeny.obs['MAPK_mlm'] = acts_progeny.obsm['score_mlm'].iloc[:, 5]
acts_progeny.obs['NFkB_mlm'] = acts_progeny.obsm['score_mlm'].iloc[:, 6]

In [ ]:
sc.pl.spatial(acts_progeny, color='EGFR_mlm', cmap='RdBu_r', size=1.3, img_key=None, vmin=-6)

In [ ]:
sc.pl.spatial(acts_progeny, color='PI3K_mlm', cmap='RdBu_r', size=1.3, img_key=None, vmin=-2.5)

In [ ]:
sc.pl.spatial(acts_progeny, color='Hypoxia_mlm', cmap='RdBu_r', size=1.3, img_key=None, vmin=-2, vmax=3)

In [ ]:
sc.pl.spatial(acts_progeny, color='JAK-STAT_mlm', cmap='RdBu_r', size=1.3, img_key=None, vmin=-3)

In [ ]:
sc.pl.spatial(acts_progeny, color='MAPK_mlm', cmap='RdBu_r', size=1.3, img_key=None, vmax=2.5, vmin=-3)

In [ ]:
sc.pl.spatial(acts_progeny, color='NFkB_mlm', cmap='RdBu_r', size=1.3, img_key=None, vmin=-2, vmax=3.5)

In [ ]:
sc.pl.spatial(acts_progeny, color=adata_8um_original_sub2.obsm['score_mlm'].columns, cmap='RdBu_r', size=1.3, img_key=None)

In [ ]:
adata_8um_original_sub2.uns['celltype_colors'] = ['#1f77b4', '#abddde', '#00ffff', '#fc6fcf', '#cc66ff', '#f297a7', '#289e92', '#ff2600', '#17becf', '#d7ef9b', 
                                     '#ffff0a', '#e6994d', '#ff9896', '#5084c2', '#80ff08']

In [ ]:
adata_8um_original_sub2

In [ ]:
acts_progeny

In [ ]:
acts_progeny

In [ ]:
sc.set_figure_params(figsize=(5, 5))
ax_list = sc.pl.violin(acts_progeny, keys=acts_progeny.obsm['score_mlm'].columns, groupby="celltype", rotation=90, show=False)
if isinstance(ax_list, list):
    for ax in ax_list:
        ax.set_ylim(-4, 6) # Set desired y-limits for each axis
else:
    # If only one plot was generated, it might return a single Axes object
    ax_list.set_ylim(-4, 6)
plt.show()

In [ ]:
acts_progeny.write_h5ad("./VisiumHD_original_acts_progeny.h5ad")